## 1. Ranked actions + reason codes

This notebook turns the validated Week 6 model output into a practical content action playbook.

The goal is not to automate content changes. The goal is to create a ranked review queue that helps a human decide which pages to inspect first.

Each row receives:

- a model score
- a rank
- a reason code
- a suggested content action
- a human review note

The output should be treated as decision support, not automatic truth.

In [1]:
!pip install -q huggingface_hub pyarrow

import os
import json
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import list_repo_files, hf_hub_download

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN is not set. Add HF_TOKEN in Colab Secrets first.")

repo_id = "FlyRank/internship-warehouse"
target_table = "fact_content_daily_performance"
target_month = "month=2026-03"

files = list_repo_files(
    repo_id,
    repo_type="dataset",
    token=HF_TOKEN
)

march_files = [
    f for f in files
    if target_table in f
    and target_month in f
    and f.endswith(".parquet")
]

print("March files found:", len(march_files))
print(march_files[:10])

March files found: 1
['fact_content_daily_performance/month=2026-03/data_0.parquet']


In [2]:
local_files = []

for file_path in march_files:
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_path,
        repo_type="dataset",
        token=HF_TOKEN
    )
    local_files.append(local_path)

df = pd.concat(
    [pd.read_parquet(path) for path in local_files],
    ignore_index=True
)

print("Rows loaded:", len(df))
print("Columns:", len(df.columns))

df.head()

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Rows loaded: 9841378
Columns: 30


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
work_df = df[df["gsc_data_available"].eq(True)].copy()
work_df = work_df[work_df["gsc_impressions"] > 0].copy()

work_df["gsc_ctr"] = np.where(
    work_df["gsc_impressions"] > 0,
    work_df["gsc_clicks"] / work_df["gsc_impressions"],
    0
)

model_df = work_df[work_df["ga4_sessions"] > 0].copy()

model_df["needs_review_proxy"] = (
    model_df["ga4_engaged_sessions"] == 0
).astype(int)

print("Rows for action playbook:", len(model_df))
print("Target distribution:")
print(model_df["needs_review_proxy"].value_counts())
print("Target rate:", round(model_df["needs_review_proxy"].mean(), 3))

Rows for action playbook: 361095
Target distribution:
needs_review_proxy
1    335986
0     25109
Name: count, dtype: int64
Target rate: 0.93


In [4]:
feature_cols = [
    "gsc_impressions",
    "gsc_avg_position",
    "gsc_ctr",
    "ga4_pageviews",
    "ga4_sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_ai",
]

X = model_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = model_df["needs_review_proxy"]
groups = model_df["client_hash_id"]

In [5]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

test_df = model_df.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", model_df.iloc[train_idx]["client_hash_id"].nunique())
print("Test clients:", model_df.iloc[test_idx]["client_hash_id"].nunique())

Train rows: 301641
Test rows: 59454
Train clients: 25
Test clients: 9


In [6]:
model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

test_df["model_score"] = model.predict_proba(X_test)[:, 1]

test_df["rank"] = test_df["model_score"].rank(
    method="first",
    ascending=False
).astype(int)

ranked_df = test_df.sort_values("model_score", ascending=False).copy()

ranked_df[
    [
        "rank",
        "content_hash_id",
        "client_hash_id",
        "report_date",
        "model_score",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_ctr",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "needs_review_proxy",
    ]
].head(20)

,rank,content_hash_id,client_hash_id,report_date,model_score,gsc_impressions,gsc_clicks,gsc_ctr,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,needs_review_proxy
9821342,9109,content_991bde97e732dfd1,client_fef1a8f436438636,2026-03-31,0.975591,90,1,0.011111,13.833333,1.0,0.0,1
15428,3,content_3c286ded8bd68120,client_65de48885f4ef01b,2026-03-01,0.975591,88,1,0.011364,8.431818,1.0,0.0,1
14480,1,content_5c80451459c29b4a,client_65de48885f4ef01b,2026-03-01,0.975591,5,0,0.000000,5.400000,1.0,0.0,1
14855,2,content_62673eea26c31c17,client_65de48885f4ef01b,2026-03-01,0.975591,3282,1,0.000305,6.167885,1.0,0.0,1
43251,10,content_66e72efd246ec696,client_3197e6291363b4db,2026-03-02,0.975591,10,1,0.100000,37.100000,1.0,0.0,1
43093,9,content_1cad01bf0c8f4a4c,client_3197e6291363b4db,2026-03-02,0.975591,243,1,0.004115,5.839506,1.0,0.0,1
9821296,9108,content_edd32d4477d028cf,client_fef1a8f436438636,2026-03-31,0.975591,4,0,0.000000,3.750000,1.0,0.0,1
6796819,6029,content_ad628fe6604aaa16,client_3197e6291363b4db,2026-03-23,0.975591,85,0,0.000000,6.470588,1.0,0.0,1
6796692,6028,content_34ba4e9a0d71f6db,client_3197e6291363b4db,2026-03-23,0.975591,57,0,0.000000,11.982456,1.0,0.0,1
6796620,6027,content_0eedd63330e9e114,client_3197e6291363b4db,2026-03-23,0.975591,19,1,0.052632,12.894737,1.0,0.0,1


In [7]:
def assign_reason_code(row):
    if row["gsc_impressions"] >= 100 and row["gsc_ctr"] == 0:
        return "visible_zero_ctr"
    elif row["gsc_impressions"] >= 100 and row["gsc_ctr"] < 0.01:
        return "visible_low_ctr"
    elif row["ga4_sessions"] > 0 and row["ga4_engaged_sessions"] == 0:
        return "traffic_no_engagement"
    elif row["gsc_avg_position"] <= 20 and row["gsc_ctr"] < 0.01:
        return "ranking_without_clicks"
    else:
        return "general_review_candidate"


def assign_action_label(reason_code):
    action_map = {
        "visible_zero_ctr": "review_title_meta_and_search_intent",
        "visible_low_ctr": "improve_title_meta_and_snippet_match",
        "traffic_no_engagement": "review_content_quality_and_engagement",
        "ranking_without_clicks": "check_serp_intent_and_page_promise",
        "general_review_candidate": "manual_content_review",
    }

    return action_map.get(reason_code, "manual_content_review")


def assign_human_review_note(reason_code):
    note_map = {
        "visible_zero_ctr": "Check whether the page appears for relevant queries and whether the title/meta description gives users a reason to click.",
        "visible_low_ctr": "Review the search snippet, query intent, and whether the page promise matches what searchers expect.",
        "traffic_no_engagement": "Check whether the page answers the visitor need, loads correctly, and has useful next steps.",
        "ranking_without_clicks": "Inspect SERP layout, intent mismatch, and whether the query may be zero-click.",
        "general_review_candidate": "Review page context before taking action.",
    }

    return note_map.get(reason_code, "Review page context before taking action.")


ranked_df["reason_code"] = ranked_df.apply(assign_reason_code, axis=1)
ranked_df["action_label"] = ranked_df["reason_code"].apply(assign_action_label)
ranked_df["human_review_note"] = ranked_df["reason_code"].apply(assign_human_review_note)

action_queue = ranked_df[
    [
        "rank",
        "content_hash_id",
        "client_hash_id",
        "report_date",
        "model_score",
        "reason_code",
        "action_label",
        "human_review_note",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_ctr",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "needs_review_proxy",
    ]
].copy()

action_queue.head(20)

,rank,content_hash_id,client_hash_id,report_date,model_score,reason_code,action_label,human_review_note,gsc_impressions,gsc_clicks,gsc_ctr,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,needs_review_proxy
9821342,9109,content_991bde97e732dfd1,client_fef1a8f436438636,2026-03-31,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,90,1,0.011111,13.833333,1.0,0.0,1
15428,3,content_3c286ded8bd68120,client_65de48885f4ef01b,2026-03-01,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,88,1,0.011364,8.431818,1.0,0.0,1
14480,1,content_5c80451459c29b4a,client_65de48885f4ef01b,2026-03-01,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,5,0,0.000000,5.400000,1.0,0.0,1
14855,2,content_62673eea26c31c17,client_65de48885f4ef01b,2026-03-01,0.975591,visible_low_ctr,improve_title_meta_and_snippet_match,"Review the search snippet, query intent, and w...",3282,1,0.000305,6.167885,1.0,0.0,1
43251,10,content_66e72efd246ec696,client_3197e6291363b4db,2026-03-02,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,10,1,0.100000,37.100000,1.0,0.0,1
43093,9,content_1cad01bf0c8f4a4c,client_3197e6291363b4db,2026-03-02,0.975591,visible_low_ctr,improve_title_meta_and_snippet_match,"Review the search snippet, query intent, and w...",243,1,0.004115,5.839506,1.0,0.0,1
9821296,9108,content_edd32d4477d028cf,client_fef1a8f436438636,2026-03-31,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,4,0,0.000000,3.750000,1.0,0.0,1
6796819,6029,content_ad628fe6604aaa16,client_3197e6291363b4db,2026-03-23,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,85,0,0.000000,6.470588,1.0,0.0,1
6796692,6028,content_34ba4e9a0d71f6db,client_3197e6291363b4db,2026-03-23,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,57,0,0.000000,11.982456,1.0,0.0,1
6796620,6027,content_0eedd63330e9e114,client_3197e6291363b4db,2026-03-23,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,19,1,0.052632,12.894737,1.0,0.0,1


### Reason code explanation

The playbook uses reason codes to explain why a page appears in the ranked queue.

- `visible_zero_ctr`: the page had search impressions but no clicks.
- `visible_low_ctr`: the page had visibility but weak click-through performance.
- `traffic_no_engagement`: the page had sessions but no measured engaged sessions.
- `ranking_without_clicks`: the page had a visible average position but weak CTR.
- `general_review_candidate`: the page needs manual inspection before deciding an action.

These reason codes are not automatic diagnoses. They are starting points for human review.

### Archetype to action mapping

| Archetype | Reason code | Suggested action | Human review question |
|---|---|---|---|
| Visible but no clicks | `visible_zero_ctr` | Review title, meta description, and search intent | Is the page appearing for relevant queries? |
| Visible but weak CTR | `visible_low_ctr` | Improve title/meta and snippet match | Does the search result promise match the page content? |
| Traffic but no engagement | `traffic_no_engagement` | Review content quality and page experience | Does the page answer the visitor's need quickly? |
| Ranking without clicks | `ranking_without_clicks` | Check SERP intent and page promise | Is this query likely zero-click or mismatched? |
| General candidate | `general_review_candidate` | Manual content review | Is there enough context to justify a change? |

### Decay / refresh insight

The playbook is based on a content-refresh idea: some pages may still receive impressions or traffic, but their observed engagement or click behavior suggests they may deserve review.

A page with impressions but no clicks may need title, meta, or search-intent review. A page with sessions but no engagement may need content quality or page experience review.

This does not prove that the page is decaying or that refreshing it will recover performance. It only suggests that the page may deserve human review based on observed search and engagement signals.

## 2. Intended use and limits

### Intended use

This playbook is intended for content and SEO review planning.

The ranked queue helps a human reviewer decide which pages to inspect first. It can support weekly or monthly content review meetings by showing pages with observed search visibility, weak click behavior, or weak engagement behavior.

The intended user is a human reviewer, such as a content strategist, SEO analyst, editor, or marketing team member.

The intended action is not automatic publishing. The intended action is review, diagnosis, and prioritization.

### Limits

This playbook has important limits:

- The target is a proxy, not a direct label of true content refresh need.
- The model does not prove that a page needs a refresh.
- The model does not prove causation.
- The model does not know the actual page text, title, meta description, SERP layout, query intent, or business value.
- The ranked queue should not be used to automatically rewrite, prune, redirect, or publish content.
- A human reviewer must inspect the page before taking action.

Safe claim:
This playbook ranks pages that may deserve review based on observed search and engagement signals. It is decision support, not automatic truth.

## 3. Human review + the no-go list

### Human review rules

Before taking action on a ranked page, a human reviewer should check:

1. Is the page important to the business?
2. What queries or user intents may be driving the impressions?
3. Is low CTR caused by the title/meta description, or by SERP layout?
4. Does the page content still match the search intent?
5. Is the content outdated, thin, duplicated, or missing useful information?
6. Is the page supposed to drive clicks, or is it naturally zero-click?
7. Is the page technically working and mobile-friendly?
8. Does the page have enough traffic or business value to justify editing?
9. Could editing the page harm a currently useful page?
10. Should the page be refreshed, protected, monitored, merged, or left alone?

### What should NOT be automated

The following actions should not be automated by this model:

- Automatically rewriting content
- Automatically publishing edited pages
- Automatically deleting or pruning pages
- Automatically redirecting pages
- Automatically changing titles or meta descriptions
- Automatically deciding that a page is bad
- Automatically deciding that a refresh caused recovery
- Automatically using the score as the only prioritization factor

The model output should only create a review queue. A human should make the final content decision.

### Cost and value thinking

The value of this playbook is prioritization. It helps a team avoid reviewing pages randomly.

High-value use:
A reviewer starts with the top-ranked pages, checks the reason codes, and decides which pages deserve action.

Cost of a false positive:
The team may spend time reviewing a page that does not need changes.

Cost of a false negative:
A page that may deserve review may be missed or delayed.

Because the cost of a wrong action can be higher than the cost of review, the playbook should recommend inspection, not automatic changes.

## 4. Monitoring / retrain triggers

This playbook should be monitored over time.

Light monitoring triggers:

- The number of reviewed pages with useful actions drops sharply.
- Many top-ranked pages are judged irrelevant by human reviewers.
- The model keeps ranking the same type of low-value page.
- Search behavior changes after a major algorithm update.
- Tracking changes affect GSC or GA4 fields.
- A large number of new clients or pages enter the dataset.
- The target proxy rate changes strongly compared with the earlier validation period.

Retrain or redesign triggers:

- Human reviewers no longer trust the ranked queue.
- The proxy target no longer matches the review goal.
- New features become available, such as page title, meta description, content age, content type, or query-level data.
- The model performs much worse on a newer time period or unseen client group.
- A better validation design becomes available.

This is a non-production playbook. Before production use, it would need stronger validation, clearer labels, reviewer feedback loops, monitoring dashboards, and documented approval rules.

In [8]:
output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

queue_path = os.path.join(output_dir, "w07_ranked_action_queue.csv")

action_queue.to_csv(queue_path, index=False)

print("Wrote:", queue_path)
print("Rows exported:", len(action_queue))

action_queue.head(10)

Wrote: work/outputs/w07_ranked_action_queue.csv
Rows exported: 59454


,rank,content_hash_id,client_hash_id,report_date,model_score,reason_code,action_label,human_review_note,gsc_impressions,gsc_clicks,gsc_ctr,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,needs_review_proxy
9821342,9109,content_991bde97e732dfd1,client_fef1a8f436438636,2026-03-31,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,90,1,0.011111,13.833333,1.0,0.0,1
15428,3,content_3c286ded8bd68120,client_65de48885f4ef01b,2026-03-01,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,88,1,0.011364,8.431818,1.0,0.0,1
14480,1,content_5c80451459c29b4a,client_65de48885f4ef01b,2026-03-01,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,5,0,0.000000,5.400000,1.0,0.0,1
14855,2,content_62673eea26c31c17,client_65de48885f4ef01b,2026-03-01,0.975591,visible_low_ctr,improve_title_meta_and_snippet_match,"Review the search snippet, query intent, and w...",3282,1,0.000305,6.167885,1.0,0.0,1
43251,10,content_66e72efd246ec696,client_3197e6291363b4db,2026-03-02,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,10,1,0.100000,37.100000,1.0,0.0,1
43093,9,content_1cad01bf0c8f4a4c,client_3197e6291363b4db,2026-03-02,0.975591,visible_low_ctr,improve_title_meta_and_snippet_match,"Review the search snippet, query intent, and w...",243,1,0.004115,5.839506,1.0,0.0,1
9821296,9108,content_edd32d4477d028cf,client_fef1a8f436438636,2026-03-31,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,4,0,0.000000,3.750000,1.0,0.0,1
6796819,6029,content_ad628fe6604aaa16,client_3197e6291363b4db,2026-03-23,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,85,0,0.000000,6.470588,1.0,0.0,1
6796692,6028,content_34ba4e9a0d71f6db,client_3197e6291363b4db,2026-03-23,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,57,0,0.000000,11.982456,1.0,0.0,1
6796620,6027,content_0eedd63330e9e114,client_3197e6291363b4db,2026-03-23,0.975591,traffic_no_engagement,review_content_quality_and_engagement,Check whether the page answers the visitor nee...,19,1,0.052632,12.894737,1.0,0.0,1


In [9]:
reason_code_counts = (
    action_queue["reason_code"]
    .value_counts()
    .to_dict()
)

action_label_counts = (
    action_queue["action_label"]
    .value_counts()
    .to_dict()
)

w07_metrics = {
    "notebook": "w07_action_playbook",
    "lane": "Lane 2 - Refresh / Content Opportunity Scoring",
    "split": "client_hash_id grouped split",
    "model": "DecisionTreeClassifier",
    "rows_in_action_queue": int(len(action_queue)),
    "top_reason_codes": reason_code_counts,
    "top_action_labels": action_label_counts,
    "intended_use": "decision support for human content review",
    "automation_level": "ranked review queue only; no automatic content changes",
}

metrics_path = os.path.join(output_dir, "w07_action_playbook_metrics.json")

with open(metrics_path, "w") as f:
    json.dump(w07_metrics, f, indent=2)

print("Wrote:", metrics_path)
w07_metrics

Wrote: work/outputs/w07_action_playbook_metrics.json


{'notebook': 'w07_action_playbook',
 'lane': 'Lane 2 - Refresh / Content Opportunity Scoring',
 'split': 'client_hash_id grouped split',
 'model': 'DecisionTreeClassifier',
 'rows_in_action_queue': 59454,
 'top_reason_codes': {'traffic_no_engagement': 38996,
  'visible_low_ctr': 11845,
  'visible_zero_ctr': 6272,
  'general_review_candidate': 1794,
  'ranking_without_clicks': 547},
 'top_action_labels': {'review_content_quality_and_engagement': 38996,
  'improve_title_meta_and_snippet_match': 11845,
  'review_title_meta_and_search_intent': 6272,
  'manual_content_review': 1794,
  'check_serp_intent_and_page_promise': 547},
 'intended_use': 'decision support for human content review',
 'automation_level': 'ranked review queue only; no automatic content changes'}

## 5. Exports for the paper

The notebook exports a ranked action queue to `work/outputs/w07_ranked_action_queue.csv`.

This queue is used as a working artifact for the recommendations section of the research paper. It stays out of Git by design because it is generated data.

The notebook also exports `work/outputs/w07_action_playbook_metrics.json` as a lightweight receipt that summarizes the playbook output.

## 6. Self-check

- [x] I created ranked actions from the validated model output.
- [x] I added reason codes.
- [x] I mapped reason codes to suggested human actions.
- [x] I included archetype-to-action mapping.
- [x] I explained the decay/refresh insight carefully.
- [x] I explained intended use and limits.
- [x] I included human review rules.
- [x] I stated what should not be automated.
- [x] I included cost/value thinking.
- [x] I included monitoring and retrain triggers.
- [x] I exported the ranked queue to `work/outputs/`.
- [x] I kept the playbook practical and non-production.
- [x] I used safe language such as observed, measured, proxy, ranked queue, human review, and decision support.